# Image Caption Generator

The goal of this project is to build an algorithm that can generate a caption based on an image.

## 1.0 Setup



### Importing the libraries

In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

2025-07-06 23:05:59.863296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751832359.983485 2288203 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751832360.014121 2288203 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751832360.233380 2288203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751832360.233407 2288203 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751832360.233410 2288203 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import string
from PIL import Image
import os
from pickle import dump, load
import numpy as np
import pandas
import keras
import sys
import h5py

sys.path.append(os.path.abspath("..")) 

from src import *

# small library for seeing the progress of loops.
from tqdm import tqdm_notebook as tqdm
tqdm().pandas()

0it [00:00, ?it/s]

/tmp/ipykernel_2288203/1592450195.py:17: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  tqdm().pandas()


0it [00:00, ?it/s]

### Setting up paths variables

In [3]:
# Paths to the folders and files
TOKEN_PATH = text_data_raw_dir / "Flickr8k.token.txt"

In [4]:
TOKEN_PATH

PosixPath('/home/mahmoud/Documents/Software/ML/img_caption_generator/data/raw/Flickr8k_text/Flickr8k.token.txt')

In [5]:
IMG_PATH = img_data_raw_dir

In [6]:
IMG_PATH

PosixPath('/home/mahmoud/Documents/Software/ML/img_caption_generator/data/raw/Flickr8k_Dataset/Flicker8k_Dataset')

## Testing Functions

In [7]:
# Create function to load the files
def load_file(filename:str):
    """
    load_file: takes a filename and loads its contents into a string

    args: 
        filename (string): path of the file that will be loaded

    returns:
        text (string): a string of all of the lines in the filename
    """

    file = open(filename, 'r')
    text = file.read()
    file.close()
    
    return text

In [8]:
# Create a function to seperate each image with its captions
def img_captions(filename:str):
    """
    img_captions: takes a file of images and all of its captions and creates a dictionary with each image 
                  as the key and the values are all of its  captions

    args:
        filename (string): the name of the file with the images and captions pair

    returns:
        image_descriptions (dictionary): a dictionary with each image as the key and its captions in a list as the value
    """

    # loading the file
    file = load_file(filename)
    captions = file.split("\n")
    image_captions = {}

    for caption in captions[:-1]: # Last line is empty so iterating up to the line before the end
        img, caption = caption.split('\t') # Splitting the image and caption using the 'tab'
        if img[:-2] not in image_captions: # Last 2 in every image has the "#" and "caption number" 
            image_captions[img[:-2]] = [caption]
        else:
            image_captions[img[:-2]].append(caption)

    return image_captions


In [9]:
img_caps = img_captions(TOKEN_PATH)

In [10]:
import pprint
from itertools import islice
pprint.pprint(dict(islice(img_caps.items(), 3)))

{'1000268201_693b08cb0e.jpg': ['A child in a pink dress is climbing up a set '
                               'of stairs in an entry way .',
                               'A girl going into a wooden building .',
                               'A little girl climbing into a wooden playhouse '
                               '.',
                               'A little girl climbing the stairs to her '
                               'playhouse .',
                               'A little girl in a pink dress going into a '
                               'wooden cabin .'],
 '1001773457_577c3a7d70.jpg': ['A black dog and a spotted dog are fighting',
                               'A black dog and a tri-colored dog playing with '
                               'each other on the road .',
                               'A black dog and a white dog with brown spots '
                               'are staring at each other in the street .',
                               'Two dogs of differ

In [11]:
# Create a function to simplify captions
def simplify(captions:dict):
    """
    simplify: takes a dictionary of image and its captions and removes punctuations, numbers, and changes uppercase letters into lowercase.

    args: 
        captions (dictionary): A dictionary of each image as the key and a list of captions as the value.

    returns:
        captions (dictionary): Returns the same dictionary after simplifying the captions 
    """
    
    table = str.maketrans("", "", string.punctuation)
    
    for img, img_captions in captions.items():
        for i, caption in enumerate(img_captions):
            
            caption.replace("-"," ") # Replacing "-" with a blank space
            words = caption.split() # Splitting each word in the dictionary

            words = [word.lower() for word in words] 
            words = [word.translate(table) for word in words] # Applying the translation table of removing the punctuation marks

            words = [word for word in words if len(word)>1] # Only keeping words that are more then 1 character long
            words = [word for word in words if word.isalpha()] # Removing numbers

            caption = " ".join(words)
            captions[img][i] = caption

    return captions

In [12]:
img_caps_simplified = simplify(img_caps)

In [13]:
pprint.pprint(dict(islice(img_caps_simplified.items(), 3)))

{'1000268201_693b08cb0e.jpg': ['child in pink dress is climbing up set of '
                               'stairs in an entry way',
                               'girl going into wooden building',
                               'little girl climbing into wooden playhouse',
                               'little girl climbing the stairs to her '
                               'playhouse',
                               'little girl in pink dress going into wooden '
                               'cabin'],
 '1001773457_577c3a7d70.jpg': ['black dog and spotted dog are fighting',
                               'black dog and tricolored dog playing with each '
                               'other on the road',
                               'black dog and white dog with brown spots are '
                               'staring at each other in the street',
                               'two dogs of different breeds looking at each '
                               'other on the road',
  

In [14]:
# Create a function to make up the vocab used in the text
def make_vocab(captions:dict):
    """
    make_vocab: takes a dictionary of image and its captions and finds all unique words used to make up a set of vocabulary

    args:
        captions (dictionary): A dictionary of each image as the key and a list of captions as the value.

    returns:
        vocab (set): A set of unique words from all the captions
    """

    vocab = set()

    for key in captions.keys():
        
        [vocab.update(c.split()) for c in captions[key]]

    return vocab   

In [15]:
vocab = make_vocab(img_caps_simplified)

In [16]:
len(vocab), list(vocab)[:10]

(8763,
 ['emerge',
  'right',
  'oak',
  'at',
  'pecking',
  'bikins',
  'daytime',
  'kiss',
  'experiences',
  'roller'])

In [17]:
# Create a function that saves the images and its new editted captions
def save_captions(captions:dict, filename:str):
    """
    save_captions: takes the new dictionary of image and its new editted captions and saves them 
                   in the original formate of image - caption_number as a text file

    args:
        captions (dictionary): A dictionary of each image as the key and a list of captions as 
                               the value.
        
        filename (string): The filename of the text that will be saved.
    """

    lines = []
    for img, image_captions in captions.items():
        for i, caption in enumerate(image_captions):
            line = img + "\t" + caption
            lines.append(line)

    text ="\n".join(lines)
    
    file = open(filename, "w")
    file.write(text)
    file.close()

In [18]:
save_captions(img_caps_simplified, data_cleaned_dir / "Flickr8k.token_cleaned.txt")

## Building a base model for feature extraction

Transfer learning using the Xception model as the base model to extract features
After removing the top layers from the Xception model which is trained on the imagenet dataset in order to extract the feature vector for the images in our dataset 

In [19]:
xception_feature_extraction_model = keras.applications.Xception(include_top=False, pooling='average') # Don't want the final layers which output the classification prediction instead we stop at the raw features detected
features = {}
for img in tqdm(os.listdir(IMG_PATH)):
    filename = IMG_PATH / img
    image = Image.open(filename)
    image = image.resize((299, 299))
    image = np.expand_dims(image, axis=0) # changing the shape of the image to (1, 299, 299, 3) as keras expect the input in batches even if batch of 1.
    image = image / 127.5
    image = image - 1.0
    
    feature = xception_feature_extraction_model.predict(image) # outputs the feature vector
    features[img] = feature
    

I0000 00:00:1751832365.690140 2288203 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5253 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
/tmp/ipykernel_2288203/2355951555.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for img in tqdm(os.listdir(IMG_PATH)):


  0%|          | 0/8091 [00:00<?, ?it/s]

I0000 00:00:1751832369.974956 2288288 service.cc:152] XLA service 0x7f68dc002f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751832369.974996 2288288 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-07-06 23:06:10.043497: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1751832370.873115 2288288 cuda_dnn.cc:529] Loaded cuDNN version 90300


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


I0000 00:00:1751832375.749984 2288288 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━

In [20]:
pprint.pprint(dict(islice(features.items(), 1)))

{'3435015880_eda46ff50f.jpg': array([[[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.05067283, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],

        [[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [1.1557364 , 0.19006318, 0.        , ..., 0.     

In [21]:
# Saving the features vector 
# dump(features, open("../models/features.p","wb"))

`dump()` kept crashing the kernel, after testing it seems to be an issue with how large the `features` dictionary is, so had to save the features in chunks one at a time

In [22]:
with h5py.File('../models/features.h5', 'w') as f:
    for k, v in features.items():
        f.create_dataset(k, data=np.array(v), compression="gzip")


Loading the file was also crashing the kernel so, had to load each vector iteratively.

In [23]:

with h5py.File('../models/features.h5', 'r') as f:
    for i, k in enumerate(f.keys()):
        features[k] = f[k][()]  # Load individual array
        if i % 100 == 0:
            print(f"Loaded {i} items...")  # Debug progress

Loaded 0 items...
Loaded 100 items...
Loaded 200 items...
Loaded 300 items...
Loaded 400 items...
Loaded 500 items...
Loaded 600 items...
Loaded 700 items...
Loaded 800 items...
Loaded 900 items...
Loaded 1000 items...
Loaded 1100 items...
Loaded 1200 items...
Loaded 1300 items...
Loaded 1400 items...
Loaded 1500 items...
Loaded 1600 items...
Loaded 1700 items...
Loaded 1800 items...
Loaded 1900 items...
Loaded 2000 items...
Loaded 2100 items...
Loaded 2200 items...
Loaded 2300 items...
Loaded 2400 items...
Loaded 2500 items...
Loaded 2600 items...
Loaded 2700 items...
Loaded 2800 items...
Loaded 2900 items...
Loaded 3000 items...
Loaded 3100 items...
Loaded 3200 items...
Loaded 3300 items...
Loaded 3400 items...
Loaded 3500 items...
Loaded 3600 items...
Loaded 3700 items...
Loaded 3800 items...
Loaded 3900 items...
Loaded 4000 items...
Loaded 4100 items...
Loaded 4200 items...
Loaded 4300 items...
Loaded 4400 items...
Loaded 4500 items...
Loaded 4600 items...
Loaded 4700 items...
Load

In [24]:
pprint.pprint(dict(islice(features.items(), 1)))

{'3435015880_eda46ff50f.jpg': array([[[[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [0.05067283, 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ]],

        [[0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         [0.        , 0.        , 0.        , ..., 0.        ,
          0.        , 0.        ],
         ...,
         [1.1557364 , 0.19006318, 0.        , ..., 0.     